# L5c Supporting Algorithm: Revised Simplex

A resource-allocation model can have many feasible choices. Once we have written its linear objective and constraints, how does a solver decide which variables to change and when to stop? The revised simplex method answers these questions by working with a basis: a selected set of constraint columns that determines the current basic solution.

> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> - **Construct a basic feasible solution:** Use a basis of the augmented constraint matrix to determine the basic variables, and explain how feasibility and degeneracy relate to a corner of the feasible region.
> - **Trace a revised-simplex pivot:** Use reduced costs, a feasible direction, and the ratio test to select entering and leaving variables, update the basis, and distinguish the algorithm's stopping conditions.
> - **Explain computational performance:** Distinguish the number of pivots from the work per pivot, and explain how basis factorizations and sparsity contribute to practical efficiency despite an exponential worst case for Dantzig's pivot rule.

In this supporting notebook, we develop these calculations, collect them into compact pseudocode, and work through a numerical pivot. This selected deeper dive supports the [linear programming lecture](CHEME-5800-L5c-Lecture-LinearProgramming-Fall-2026.ipynb) and the [fruit-allocation example](CHEME-5800-L5c-Example-FruitAllocation-Fall-2026.ipynb). Students are not expected to implement a production linear programming solver this week.


___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file defines local paths, activates the course environment, and loads the required packages and course library.

We retain the shared Week 5 setup here; the algorithm discussion and worked pivot are mathematical, and this notebook does not call the solver.

Let's set up our code environment:


In [1]:
include(joinpath(@__DIR__, "Include.jl"))


See the [Julia documentation](https://docs.julialang.org/en/v1/) and the [course library documentation](../../../docs/src/index.md) for the shared computational environment.

___

## Revised simplex

The simplex method, developed by George Dantzig in 1947, solves a linear program by exchanging basic and nonbasic variables. Starting from a basic feasible solution, the primal method maintains feasibility while seeking a better objective value. A degenerate pivot can change the basis without changing the solution or its objective value.

> __Myth or fact?__ As a graduate student in 1939, Dantzig once arrived late to a statistics lecture, mistook two well-known unsolved problems on the blackboard for homework, and solved them over the next few days, before realizing they were open research questions! That story is true, but it happened years before he developed the simplex method and did not directly inspire the algorithm. But still, it's a fun story; being late isn't always bad!

__Simplex is a big deal__: Before simplex, LPs were mostly a theoretical curiosity; afterwards, they became essential tools, radically improving resource allocation and strategic planning in the second half of the twentieth century. Over seventy years later, the simplex method remains widely used in commercial optimization software. This is despite some not-so-great worst-case performance bounds!

### Bases and an initial feasible solution

We are going to focus on the _revised simplex algorithm_. The method partitions the variables into a __basic set__ and a __nonbasic set__. We hold the nonbasic variables at zero and determine the basic variables from the constraints. Changing which variables are basic allows us to search for a solution with a lower objective value. Let's first examine how this partition defines a feasible starting point.

Suppose we have $n$ decision variables collected in $x$, $m$ inequality constraints with coefficient matrix $A\in\mathbb{R}^{m\times n}$ and right-hand side $b\in\mathbb{R}^{m}$, and objective coefficients $c\in\mathbb{R}^{n}$. We assume $b\geq 0$. Introducing a vector of slack variables $s\in\mathbb{R}^{m}$ gives the minimization problem:

$$
\begin{aligned}
\text{minimize}\quad & c^\top x \\
\text{subject to}\quad & Ax+s=b, \\
& x\geq 0,\quad s\geq 0.
\end{aligned}
$$

Each slack variable measures the difference between a constraint's upper bound and its left-hand side. Thus, $Ax+s=b$ with $s\geq 0$ is equivalent to $Ax\leq b$. If the original problem maximizes an objective, we negate its coefficients to obtain this minimization form.

Both decision variables and slack variables can enter the basis, so our matrix must contain columns for both. Collect the variables, constraint columns, and objective coefficients as follows:

$$
z=\begin{bmatrix}x\\s\end{bmatrix},\qquad
\widetilde A=\begin{bmatrix}A&I_m\end{bmatrix},\qquad
\widetilde c=\begin{bmatrix}c\\0_m\end{bmatrix},
$$

where $I_m$ is the $m\times m$ identity matrix and $0_m$ is a vector of $m$ zeros. The augmented matrix $\widetilde A$ has $n+m$ columns. The slack variables have zero objective coefficients, so the constraints become $\widetilde A z=b$ with $z\geq 0$, and the objective remains $\widetilde c^\top z=c^\top x$.

> __How does a basis identify a corner?__
>
> Choose $m$ linearly independent columns of $\widetilde A$. Let $B=(B_1,\ldots,B_m)$ be their ordered list of column indices, and let $N$ contain the remaining indices. The selected columns form the invertible __basis matrix__ $\widetilde A_B$. Setting the nonbasic variables to zero determines the basic variables through the linear system:
>
> $$
> z_N=0,\qquad \widetilde A_B z_B=b.
> $$
>
> If the resulting basic values satisfy $z_B\geq 0$, we have a __basic feasible solution__, which represents a corner of the feasible region. A choice of independent columns alone does not guarantee feasibility; the solution must also satisfy nonnegativity.
>
> A basic variable can be zero. When at least one basic variable is zero, the basic feasible solution is __degenerate__, and different bases may represent the same corner. Thus, exchanging basic and nonbasic variables need not move us to a different point or improve the objective.

This distinction separates the variables we solve for from the values they take. The basis selects the columns used in the linear system; feasibility depends on the resulting values.

__Initialization.__ The slack columns give us a convenient first basis. With the entries of $z$ ordered as above, choose the initial indices as follows:

$$
B=(n+1,\ldots,n+m),\qquad N=\{1,\ldots,n\}.
$$

The initial basis matrix is $\widetilde A_B=I_m$. Setting the decision variables to zero and solving for the slacks gives:

$$
x^{(0)}=0,\qquad s^{(0)}=b.
$$

Because $b\geq 0$, this starting solution is feasible. If any component of $b$ is zero, the initial basic feasible solution is degenerate. This initialization relies on the stated inequality form and nonnegative right-hand side; other formulations may require a separate procedure to find a feasible basis.

Set the pivot counter $t=0$ and choose a positive integer limit $T$ on the number of pivots. We can now examine how reduced costs identify a candidate entering variable and how the ratio test determines the feasible step.


### Reduced costs and feasible steps

Starting from a basic feasible solution, we want to determine whether increasing a nonbasic variable can lower the objective. Its objective coefficient alone does not answer this question: increasing that variable also requires changes in the basic variables to maintain the equality constraints. The __reduced cost__ accounts for both contributions.

Let $\lambda\in\mathbb{R}^{m}$ be the multipliers associated with the current basis, and let $\mu_i$ be the reduced cost of nonbasic variable $z_i$. We calculate these quantities by solving for $\lambda$ and then evaluating each reduced cost:

$$
\begin{aligned}
\widetilde A_B^\top\lambda&=\widetilde c_B,\\
\mu_i&=\widetilde c_i-\lambda^\top\widetilde A_i,\qquad i\in N,
\end{aligned}
$$

where $\widetilde A_i$ is column $i$ of the augmented constraint matrix and $\widetilde c_B$ contains the objective coefficients of the current basic variables. We write the multiplier calculation as a linear system; an explicit matrix inverse is not required.

__Optimality test.__ Using $\widetilde A z=b$ and the multiplier equation, we can express the objective of any feasible solution in terms of its nonbasic variables:

$$
\widetilde c^\top z
=\lambda^\top b+\sum_{i\in N}\mu_i z_i.
$$

At the current basic feasible solution, $z_N=0$, so its objective is $\lambda^\top b$. If every nonbasic reduced cost is nonnegative, the sum cannot be negative at any feasible solution because $z_i\geq 0$. The current solution therefore attains the minimum, and we stop with an __optimal__ status.

Otherwise, choose an entering index $e$ with the most negative reduced cost. This is __Dantzig's entering-variable rule__, written as:

$$
e\in\underset{i\in N}{\operatorname{arg\,min}}\;\mu_i,\qquad \mu_e<0.
$$

We choose one minimizing index if there is a tie. A negative reduced cost identifies a candidate direction; we still need to determine whether a positive step is feasible.

__Direction and objective change.__ Let $\alpha\geq 0$ be the proposed increase in the entering variable, which currently has value zero. To determine the corresponding change $-\alpha d$ in the basic variables, solve the following linear system for $d\in\mathbb{R}^{m}$:

$$
\widetilde A_B d=\widetilde A_e.
$$

If we increase $z_e$ by $\alpha$, the current basic variables must change as follows:

$$
z_e(\alpha)=\alpha,\qquad z_B(\alpha)=z_B-\alpha d.
$$

All other nonbasic variables remain zero. These changes preserve the equality constraints because $\widetilde A_B(z_B-\alpha d)+\widetilde A_e\alpha=b$. The change in the objective is therefore:

$$
\begin{aligned}
\Delta f
&=\alpha\widetilde c_e-\alpha\widetilde c_B^\top d\\
&=\alpha\bigl(\widetilde c_e-\lambda^\top\widetilde A_e\bigr)\\
&=\alpha\mu_e.
\end{aligned}
$$

Thus, the objective strictly decreases when $\mu_e<0$ and the feasible step satisfies $\alpha>0$. A zero step leaves the objective unchanged.

__Ratio test.__ We must also keep the basic variables nonnegative. For a position $j$ in the ordered basis, a positive component $d_j$ makes $(z_B)_j$ decrease as $\alpha$ increases. That variable reaches zero at $\alpha=(z_B)_j/d_j$. When at least one $d_j>0$, the largest feasible step is the first of these limits:

$$
\alpha^\star=\min_{j:\,d_j>0}\frac{(z_B)_j}{d_j},\qquad
\ell\in\underset{j:\,d_j>0}{\operatorname{arg\,min}}\;\frac{(z_B)_j}{d_j}.
$$

The index $\ell$ is a position in the current basis; the corresponding variable $z_{B_\ell}$ leaves the basis. Components with $d_j\leq 0$ impose no upper limit on the step because those basic values remain constant or increase.

If no component of $d$ is positive, every $\alpha\geq 0$ is feasible along this direction. Since $\mu_e<0$, the objective decreases without bound, and we stop with an __unbounded__ status.

If the ratio test gives $\alpha^\star=0$, the pivot is __degenerate__: the basis changes, but the solution and objective do not. This is why a negative reduced cost alone does not establish that a degenerate current solution is nonoptimal. With the entering variable, leaving position, and step determined, we can now update the solution and basis in the correct order.


### Pivot and stopping conditions

Suppose the ratio test gives a finite step $\alpha^\star$ and a leaving position $\ell$. We must update the solution using the __current__ basis before replacing its leaving index. Otherwise, the entries of the direction vector $d$ would be paired with the wrong variables.

First, save the current basic and nonbasic indices and the index $r$ of the leaving variable:

$$
B^{\mathrm{old}}\gets B,\qquad
N^{\mathrm{old}}\gets N,\qquad
r\gets B^{\mathrm{old}}_\ell.
$$

The vector $z^{(t)}$ is the current solution after $t$ completed pivots. Form the next solution using the saved indices:

$$
\begin{aligned}
z_{B^{\mathrm{old}}}^{(t+1)}&=z_{B^{\mathrm{old}}}^{(t)}-\alpha^\star d,\\
z_e^{(t+1)}&=\alpha^\star,\\
z_i^{(t+1)}&=0,\qquad i\in N^{\mathrm{old}}\setminus\{e\}.
\end{aligned}
$$

The ratio test makes the leaving variable satisfy $z_r^{(t+1)}=0$, while preserving nonnegativity of every variable. We can therefore hold $z_r$ at zero as a nonbasic variable and replace its basis column with the entering column. Update the ordered basis, nonbasic set, and counter as follows:

$$
\begin{aligned}
B_j&\gets
\begin{cases}
e,&j=\ell,\\
B_j^{\mathrm{old}},&j\ne\ell,
\end{cases}
\qquad j=1,\ldots,m,\\[4pt]
N&\gets\bigl(N^{\mathrm{old}}\setminus\{e\}\bigr)\cup\{r\},\\
t&\gets t+1.
\end{aligned}
$$

The positive pivot component $d_\ell$ ensures that the new basis matrix is invertible. The next iteration uses this new basis to recompute the multipliers and reduced costs. A degenerate pivot follows the same update: when $\alpha^\star=0$, the basis changes even though the solution does not.

__Stopping conditions.__ At the start of each iteration, test the newly computed reduced costs for optimality. If the test fails and $t=T$, stop because the allowed number of pivots has been reached. Otherwise, choose the entering variable and compute its direction. An unbounded direction ends the calculation; a finite ratio-test step leads to the next pivot.

> __What does the returned status mean?__
>
> * **Optimal:** The current solution is feasible and every nonbasic reduced cost is nonnegative. Together, these conditions certify that the objective has reached its minimum.
> * **Unbounded:** A variable with negative reduced cost has a direction with $d_j\leq 0$ for every basic position $j$. We can increase that variable indefinitely while remaining feasible, driving the objective toward $-\infty$.
> * **Iteration limit:** We have completed $T$ pivots without obtaining an optimality certificate. The current solution is feasible, but whether it is optimal remains unresolved.

Testing optimality before the pivot limit allows us to recognize a solution that becomes optimal on the final permitted pivot. Reaching the limit alone does not establish convergence.

__Degeneracy and cycling.__ A sequence of zero-step pivots can revisit a previous basis, a behavior called _cycling_. Dantzig's entering-variable rule does not prevent this, so the iteration limit may be reached without finding an optimality certificate. One way to prevent cycling is [Bland's rule](https://ocw.mit.edu/courses/6-251j-introduction-to-mathematical-programming-fall-2009/2ef2f1dd7045b5f29e5faea299fb0798_MIT6_251JF09_lec06.pdf): choose the smallest-index nonbasic variable with negative reduced cost and, among tied minimum ratios, choose the basic variable with the smallest variable index to leave. This changes both selection rules and guarantees finite termination in exact arithmetic. Our sketch retains Dantzig's entering rule; Bland's rule explains how an anti-cycling safeguard can change the selection procedure.

We now have the complete sequence: test optimality, determine a feasible step or identify unboundedness, and update the solution and basis while tracking the number of pivots.


### Revised simplex pseudocode

Let's sketch out the revised simplex algorithm:

__Initialization__: Given the linear program $\min\{c^\top x\mid Ax+s=b,\;x\geq 0,\;s\geq 0\}$ with $b\geq 0$, use the augmented variable vector $z$, constraint matrix $\widetilde A$, and objective coefficients $\widetilde c$ defined above.

Choose the initial basis $B=(n+1,\ldots,n+m)$ and nonbasic set $N=\{1,\ldots,n\}$, with $x^{(0)}=0$ and $s^{(0)}=b$. Set the pivot counter $t\gets 0$, choose a positive integer pivot limit $T$, and set $\texttt{converged}\gets\texttt{false}$.

While not $\texttt{converged}$ __do__:

1. __Optimality test__. Solve $\widetilde A_B^\top\lambda=\widetilde c_B$ for the multipliers, then compute the reduced cost $\mu_i=\widetilde c_i-\lambda^\top\widetilde A_i$ for each nonbasic variable $i\in N$.
    - If $\mu_i\geq 0$ for all $i\in N$, set $\texttt{converged}\gets\texttt{true}$ and return the current solution $z^{(t)}$ with status __optimal__.
    - If the optimality test fails and $t=T$, return the current feasible solution $z^{(t)}$ with status __iteration limit__. Otherwise, continue to the direction and ratio test.
2. __Direction and ratio test__. Select $e\in\arg\min_{i\in N}\mu_i$, choosing the smallest variable index among tied minima. The variable $z_e$ will enter the basis. Solve $\widetilde A_B d=\widetilde A_e$ for the direction $d$.
    - If $\{j\mid d_j>0\}=\emptyset$, return the current feasible solution $z^{(t)}$ with status __unbounded__.
    - Compute the step size $\alpha^\star=\min\left\{\frac{(z_B^{(t)})_j}{d_j}\mid d_j>0\right\}$.
    - Choose a leaving position $\ell\in\arg\min_{j:\,d_j>0}\frac{(z_B^{(t)})_j}{d_j}$. Among tied minimum ratios, choose the position whose basic variable has the smallest index $B_\ell$.
3. __Pivot and update__. Save copies $B^{\mathrm{old}}\gets B$ and $N^{\mathrm{old}}\gets N$, and record the leaving variable index $r\gets B^{\mathrm{old}}_\ell$. Start the next solution as a copy, $z^{(t+1)}\gets z^{(t)}$.
    - Update the old basic values, $z_{B^{\mathrm{old}}}^{(t+1)}\gets z_{B^{\mathrm{old}}}^{(t)}-\alpha^\star d$, and set the entering value $z_e^{(t+1)}\gets\alpha^\star$. The ratio test makes the leaving value $z_r^{(t+1)}=0$.
    - Replace $B_\ell\gets e$, keeping the other basis positions unchanged, and update $N\gets(N^{\mathrm{old}}\setminus\{e\})\cup\{r\}$. Increment the counter $t\gets t+1$.
4. __Check convergence__. Loop back to step 1 to recompute the multipliers and reduced costs for the new basis. Test optimality before checking the pivot limit, including after the final permitted pivot.

A zero-step pivot changes the basis without changing the solution. These exact-arithmetic steps use Dantzig's entering rule; the tie choices make the calculation reproducible, while the pivot limit bounds the work if cycling occurs.


### A worked pivot

Let's follow one pivot for a small problem with two decision variables and two constraints. This will show how the ratio test selects a leaving variable and why we must check reduced costs again after updating the basis. Consider the minimization problem:

$$
\begin{aligned}
\text{minimize}\quad & -3x_1-2x_2\\
\text{subject to}\quad & x_1+x_2\leq 4,\\
& 2x_1+x_2\leq 5,\\
& x_1,x_2\geq 0.
\end{aligned}
$$

Introduce slack variables $s_1$ and $s_2$ and order the variables as $z=(x_1,x_2,s_1,s_2)^\top$. The augmented matrix and objective coefficients are given by:

$$
\widetilde A=
\begin{bmatrix}
1&1&1&0\\
2&1&0&1
\end{bmatrix},\qquad
\widetilde c=(-3,-2,0,0)^\top.
$$

The initial slack basis is $B=(3,4)$, with $N=\{1,2\}$ and $z^{(0)}=(0,0,4,5)^\top$. This point is feasible and has objective value zero.

__1. Optimality test.__ The initial basis matrix is the identity, and both basic objective coefficients are zero. The multiplier equation therefore gives $\lambda=(0,0)^\top$, so the nonbasic reduced costs are $\mu_1=-3$ and $\mu_2=-2$. The initial basis fails the optimality test.

__2. Direction and ratio test.__ Dantzig's rule selects $e=1$: the variable $x_1$ enters because it has the most negative reduced cost. Solving $\widetilde A_Bd=\widetilde A_1$ gives $d=(1,2)^\top$. Thus, increasing $x_1$ by $\alpha$ decreases $s_1$ by $\alpha$ and $s_2$ by $2\alpha$. The two slacks limit the step as follows:

$$
\alpha^\star=\min\left\{\frac{4}{1},\frac{5}{2}\right\}=\frac{5}{2}.
$$

The second ratio is smaller, so $\ell=2$ and the leaving variable has index $r=B_2=4$. Thus, $s_2$ leaves the basis. Although $s_2$ starts with the larger value, it decreases twice as fast and reaches zero first.

__3. Pivot and update.__ Use the old basic order $(s_1,s_2)$ to update the slack values, then set the entering variable to the step size:

$$
\begin{aligned}
\begin{bmatrix}s_1^{(1)}\\s_2^{(1)}\end{bmatrix}
&=\begin{bmatrix}4\\5\end{bmatrix}
-\frac{5}{2}\begin{bmatrix}1\\2\end{bmatrix}
=\begin{bmatrix}3/2\\0\end{bmatrix},\\
z^{(1)}&=(5/2,0,3/2,0)^\top.
\end{aligned}
$$

Replace the second basis index with $e=1$, giving $B=(3,1)$ and $N=\{2,4\}$, and set $t=1$. The first constraint has slack $3/2$; the second is now binding, meaning it is satisfied at equality. The new objective is $-3(5/2)=-15/2$, so its change agrees with the reduced-cost calculation $\Delta f=\alpha^\star\mu_1=(5/2)(-3)=-15/2$.

__4. Check convergence.__ The new ordered basis contains the columns for $(s_1,x_1)$. Its multiplier equation and the resulting nonbasic reduced costs are given by:

$$
\begin{aligned}
\begin{bmatrix}1&0\\1&2\end{bmatrix}\lambda
&=\begin{bmatrix}0\\-3\end{bmatrix}
\quad\Longrightarrow\quad \lambda=(0,-3/2)^\top,\\
\mu_2&=-2-(0-3/2)=-1/2,\\
\mu_4&=0-(-3/2)=3/2.
\end{aligned}
$$

The reduced cost of $x_2$ is still negative. Both current basic values are positive, so a positive improving step is possible. This first pivot improved the objective but did not reach the optimum; another pivot is needed. The calculation shows why moving to a new feasible corner must be followed by a fresh optimality test.


### Computational cost and practical performance

How much work does revised simplex require? Two quantities matter: the number of pivots before termination and the computational work required for each pivot. The pivot rule influences the first quantity; the way we organize the basis calculations influences the second.

- __Number of pivots.__ Choosing the most negative reduced cost is a natural local decision, but it does not guarantee a short path to an optimal basis. Klee and Minty constructed a family of linear programs for which Dantzig's pivot rule requires exponentially many pivots as the problem dimension grows. Thus, this rule has no polynomial worst-case bound on the number of pivots, even for problems on which it terminates. See the discussion of the Klee–Minty construction in [Fearnley and Savani, *The Complexity of the Simplex Method*](https://arxiv.org/abs/1404.0605).

- __Work per pivot.__ The revised method organizes the calculation around the current basis matrix $\widetilde{\mathbf{A}}_B$. Practical implementations use a factorization of this matrix to solve the systems for the multipliers $\boldsymbol{\lambda}$ and the direction $\mathbf{d}$. Since a pivot replaces only one basis column, the factorization can be updated between occasional fresh factorizations. This avoids maintaining a full transformed simplex tableau and allows the implementation to exploit sparsity in the linear algebra. [Huangfu and Hall](https://webhomes.maths.ed.ac.uk/hall/HuHa12/) describe basis-update techniques used in practical revised-simplex implementations.

- __Practical interpretation.__ These computational choices help explain why revised simplex can solve large sparse linear programs efficiently. The benefit depends on the problem structure and the implementation; [Hall and McKinnon](https://webhomes.maths.ed.ac.uk/hall/MS-00/MS-00-015-abstract.html) show how exploiting sparsity can reduce solution time on suitable problems. Efficient linear algebra reduces the work needed to follow the chosen sequence of bases. It does not give Dantzig's pivot rule a polynomial worst-case guarantee.

For Week 5, our focus is the connection between reduced costs, feasible steps, and basis changes. Interior-point derivations are reserved for deeper study.


___


## Summary

In this notebook, we developed the revised simplex method from the definition of a basic feasible solution through the calculations needed for a pivot. The pseudocode collected those calculations into an algorithm, and the worked example showed how the entering variable, feasible step, and leaving variable determine the next basis.

> __Key Takeaways:__
>
> - **A basis determines a candidate corner:** We selected independent columns of the augmented constraint matrix, set nonbasic variables to zero, and solved for the basic variables. Nonnegative basic values established feasibility; a zero basic value identified degeneracy, so a basis change need not move to a different point.
> - **Reduced costs and the ratio test guide the pivot:** Reduced costs measured the objective change per unit increase in a nonbasic variable, while the direction and ratio test determined the feasible step. We distinguished an optimality certificate, an improving unbounded direction, and an iteration limit that stops the calculation without certifying optimality.
> - **Revised simplex organizes the work around the basis:** We used basis linear systems to calculate multipliers and directions. Updating factorizations and exploiting sparsity can reduce the work per pivot, while Dantzig's pivot rule can still require exponentially many pivots on worst-case problems.

These ideas connect the geometry of a linear program with the calculations used to solve it and provide a basis for interpreting the results in the companion allocation example.
